# NVDA / MU / MSFT 均值回归 Alpha + 风险模型报告
本报告展示短期均值回归 Alpha、SEC/新闻事件过滤、EWMA 协方差风险模型和下一交易日开盘执行的组合回测。

执行假设：信号在收盘后形成，下一交易日开盘调仓，单边交易成本 5 个基点，不使用杠杆。

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "run_alpha_research.py").exists():
    raise RuntimeError("请先在 VSCode 中打开项目根目录")

required_data = [
    PROJECT_ROOT / "data" / "raw" / f"{ticker}_daily_adjusted.parquet"
    for ticker in ("NVDA", "MU", "MSFT")
]
if all(path.exists() for path in required_data):
    for script in (
        "run_alpha_research.py",
        "run_risk_model.py",
        "run_alpha_portfolio_backtest.py",
    ):
        completed = subprocess.run(
            [sys.executable, script],
            cwd=PROJECT_ROOT,
            capture_output=True,
            text=True,
            check=False,
        )
        if completed.returncode != 0:
            raise RuntimeError(f"{script} 执行失败：\n{completed.stderr}")
        print(f"完成：{script}")
else:
    print("未找到原始行情，直接展示项目包内已经生成的最新报告。")

## 分阶段回测结果
Research 为 2016–2022，Validation 为 2023–2024，Test 为 2025 年至今。Test 已经被查看，后续不应再用它调参。

In [ ]:
summary_path = PROJECT_ROOT / "outputs" / "alpha_portfolio" / "portfolio_backtest_summary.csv"
summary = pd.read_csv(summary_path)
display_columns = [
    "Model", "Period", "TotalReturn", "CAGR",
    "AnnualVolatility", "Sharpe", "MaxDrawdown",
    "AverageGrossExposure", "AnnualizedTurnover",
]
formatted = summary[display_columns].copy()
for column in (
    "TotalReturn", "CAGR", "AnnualVolatility",
    "MaxDrawdown", "AverageGrossExposure",
):
    formatted[column] = formatted[column].map(lambda value: f"{value:.2%}")
formatted["Sharpe"] = formatted["Sharpe"].map(lambda value: f"{value:.2f}")
formatted["AnnualizedTurnover"] = formatted["AnnualizedTurnover"].map(
    lambda value: f"{value:.2f}x"
)
formatted

## 2025 年至今：净值与回撤
事件调整模型相对纯价格模型有所改善，但当前 Alpha 尚未超过使用相同风险框架、完全不看 Alpha 的逆波动基准。等权买入持有收益最高，同时承担显著更大的波动和回撤。

In [ ]:
chart_path = PROJECT_ROOT / "outputs" / "alpha_portfolio" / "test_equity_and_drawdown.png"
display(Image(filename=str(chart_path)))

## 最新风险建议
风险模型采用日频 EWMA 协方差（衰减系数 0.94）、25% 相关性收缩、10% 年化目标波动率、40% 单股上限和 100% 总仓位上限。日期 t 的建议最早只能在 t+1 开盘执行。

In [ ]:
risk_path = PROJECT_ROOT / "outputs" / "risk" / "latest_risk_report.json"
risk = json.loads(risk_path.read_text(encoding="utf-8"))
latest = pd.DataFrame(
    {
        "数值": {
            "数据日期": risk["as_of"],
            "NVDA 权重": f"{risk['recommended_weights']['NVDA']:.2%}",
            "MU 权重": f"{risk['recommended_weights']['MU']:.2%}",
            "MSFT 权重": f"{risk['recommended_weights']['MSFT']:.2%}",
            "现金权重": f"{risk['cash_weight']:.2%}",
            "预测年化波动率": f"{risk['forecast_annual_volatility']:.2%}",
            "95% 单日 VaR": f"{risk['one_day_var_95']:.2%}",
            "95% 单日 Expected Shortfall": f"{risk['one_day_expected_shortfall_95']:.2%}",
        }
    }
)
latest

## 最新相关矩阵

In [ ]:
correlation = pd.DataFrame(risk["correlation"])
correlation.round(2)

## 当前结论

- 事件过滤改善了测试期的纯价格 Alpha；
- 风险模型显著降低回撤，但平均股票仓位较低；
- 当前 Alpha 尚未证明能够战胜不使用 Alpha 的风险基准；
- 下一轮研究应只使用 Research/Validation，重点测试 5 日持有期和降低换手，不能继续针对已查看的 Test 调参。